# 2_Modelo_Preditivo

Este notebook cria um modelo de Machine Learning para prever **risco de defasagem no ano seguinte**.

> Em termos simples: para cada aluno no ano atual (ano-base), estimamos a chance de ele entrar em risco no proximo ano.

## Visao geral do fluxo
1. Preparar ambiente e parametros do experimento.
2. Carregar e padronizar dados de anos diferentes.
3. Criar a variavel alvo (target) que queremos prever.
4. Separar treino e teste com corte temporal (sem misturar passado e futuro).
5. Treinar e comparar varios modelos.
6. Avaliar desempenho com metricas e graficos.
7. Explicar quais variaveis mais influenciam a previsao.
8. Exportar artefatos para uso posterior (app/API/dashboard).

## Conceitos para iniciantes
- **Feature**: coluna usada como entrada do modelo (ex.: IAN, IDA, idade).
- **Target**: coluna que o modelo tenta prever (aqui: `risco_defasagem_t1`).
- **Probabilidade de risco**: saida do modelo entre 0 e 1.
- **Threshold**: ponto de corte para converter probabilidade em classe 0/1.


## 1) Setup

Esta secao prepara o ambiente do notebook.

### O que esta celula faz
- Importa bibliotecas de manipulacao de dados (`pandas`, `numpy`), visualizacao (`plotly`) e modelagem (`scikit-learn`, `xgboost`).
- Importa funcoes de metrica para avaliar classificacao (`accuracy`, `precision`, `recall`, `f1`, `roc_auc`, etc.).
- Define caminhos importantes do projeto (`data/`, `artifacts/`) e cria a pasta de saida se nao existir.
- Configura constantes para reproducibilidade e estrategia de treino.

### Por que isso e importante
- Centralizar imports e parametros deixa o notebook mais organizado e facil de manter.
- `RANDOM_STATE = 42` ajuda a reproduzir os mesmos resultados.
- Os limites do target (`TARGET_IAN_THRESHOLD`, `TARGET_IDA_THRESHOLD`) e o objetivo de recall (`RECALL_OBJECTIVE`) tornam a regra de negocio explicita.

### Parametros chave usados depois
- `N_FOLDS = 5`: quantidade de particoes na validacao cruzada.
- `THRESHOLD_GRID`: lista de cortes testados (de 0.10 ate 0.90).
- `RECALL_OBJECTIVE = 0.80`: prioridade do projeto e capturar casos de risco.


In [1]:
import re
import unicodedata
from pathlib import Path
import warnings
import pickle

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    classification_report,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
px.defaults.template = "plotly_white"
pd.set_option("display.max_columns", 200)


def detect_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "app.py").exists():
            return candidate
    return current


PROJECT_ROOT = detect_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "pede_consolidado_2022_2024.csv"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_FOLDS = 5
TARGET_IAN_THRESHOLD = 5.0
RECALL_OBJECTIVE = 0.80
THRESHOLD_GRID = np.round(np.linspace(0.10, 0.90, 33), 3)




## 2) Funcoes de carga e harmonizacao

Nesta etapa, o foco e transformar uma base "real" (com sujeira e variacao de nomes) em uma base consistente para modelagem.

### Principais funcoes criadas
- `normalize_column_name`: padroniza nome de colunas (minusculas, sem caracteres especiais).
- `clean_empty_strings`: transforma strings vazias em `NaN` (valor ausente).
- `coerce_numeric_series`: converte textos para numero (trata `%`, virgula decimal e textos invalidos).
- `normalize_age_series`: padroniza idade mesmo quando vem em formatos estranhos (inclusive data).
- `coalesce_columns`: junta colunas equivalentes de anos diferentes em uma unica coluna padrao.
- `standardize_gender` e `standardize_stone`: padronizam categorias para evitar duplicidade semantica.
- `extract_phase_code`: extrai codigo de fase em formato unico (`ALFA`, `FASE 1`, ...).

### Consolidacao da base
A funcao `load_and_prepare_data`:
- percorre abas da planilha,
- identifica ano da aba,
- remove linhas invalidas,
- harmoniza colunas,
- converte tipos,
- limita notas na faixa 0-10,
- seleciona colunas finais,
- concatena tudo em um unico DataFrame historico.

A funcao `load_consolidated_or_raw` tenta primeiro ler o CSV consolidado ja pronto. Se nao existir, monta a base a partir do Excel bruto.

### Resumo para iniciantes
Essa etapa e o "alicerce" do modelo. Se a padronizacao for ruim, o modelo aprende ruido em vez de padrao real.


In [2]:
def normalize_column_name(col_name: str) -> str:
    normalized = unicodedata.normalize("NFKD", str(col_name)).encode("ascii", "ignore").decode("ascii")
    normalized = re.sub(r"[^0-9a-zA-Z]+", "_", normalized).strip("_").lower()
    return normalized


def clean_empty_strings(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.strip()
    return cleaned.replace({"": np.nan, "nan": np.nan, "None": np.nan, "<NA>": np.nan})


def coerce_numeric_series(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.strip()
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace({"": np.nan, "nan": np.nan, "None": np.nan, "<NA>": np.nan, "INCLUIR": np.nan, "incluir": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")


def normalize_age_series(series: pd.Series) -> pd.Series:
    numeric_age = coerce_numeric_series(series)
    date_values = pd.to_datetime(series, errors="coerce")

    age_from_date = np.where(
        date_values.notna() & (date_values.dt.year == 1900) & (date_values.dt.month == 1),
        date_values.dt.day,
        np.nan,
    )

    result = pd.Series(numeric_age, index=series.index)
    mask = result.isna() & ~pd.isna(age_from_date)
    result.loc[mask] = age_from_date[mask]
    result = result.where(result.between(6, 30))
    return result.round()


def coalesce_columns(df: pd.DataFrame, new_col: str, candidates: list[str], numeric: bool = False) -> None:
    result = None
    for col in candidates:
        if col not in df.columns:
            continue
        values = coerce_numeric_series(df[col]) if numeric else clean_empty_strings(df[col])
        result = values if result is None else result.where(result.notna(), values)
    if result is None:
        result = pd.Series(np.nan, index=df.index)
    df[new_col] = result


def standardize_gender(value: str) -> str:
    if pd.isna(value):
        return np.nan
    value_clean = str(value).strip().lower()
    mapping = {"menino": "Masculino", "masculino": "Masculino", "menina": "Feminino", "feminino": "Feminino"}
    return mapping.get(value_clean, str(value).strip())


def standardize_stone(value: str) -> str:
    if pd.isna(value):
        return np.nan
    value_norm = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode("ascii")
    value_norm = value_norm.strip().lower()
    mapping = {
        "quartzo": "Quartzo",
        "agata": "Agata",
        "ametista": "Ametista",
        "topazio": "Topazio",
        "incluir": np.nan,
    }
    return mapping.get(value_norm, str(value).strip())


def extract_phase_code(value: str) -> str:
    if pd.isna(value):
        return np.nan
    raw = str(value).strip().upper()
    if raw in {"", "NAN", "NONE"}:
        return np.nan
    if "ALFA" in raw:
        return "ALFA"
    match = re.search(r"FASE\s*([0-9]+)", raw)
    if match:
        return f"FASE {int(match.group(1))}"
    if re.fullmatch(r"[0-9]+", raw):
        phase_num = int(raw)
        return "ALFA" if phase_num == 0 else f"FASE {phase_num}"
    return np.nan


def load_and_prepare_data(data_path: Path) -> pd.DataFrame:
    xls = pd.ExcelFile(data_path)
    all_frames = []

    for sheet in xls.sheet_names:
        year_match = re.search(r"20\d{2}", sheet)
        if not year_match:
            continue
        year = int(year_match.group())

        raw = xls.parse(sheet)
        raw = raw.dropna(how="all").copy()
        raw.columns = [normalize_column_name(c) for c in raw.columns]

        if "ra" not in raw.columns:
            continue
        raw["ra"] = clean_empty_strings(raw["ra"])
        raw = raw[~raw["ra"].str.upper().isin(["RA-1", "NAN"])]
        raw = raw[raw["ra"].notna()]
        raw["ano_referencia"] = year

        coalesce_columns(raw, "idade", ["idade", "idade_22"], numeric=False)
        coalesce_columns(raw, "instituicao_ensino", ["instituicao_de_ensino", "escola"], numeric=False)
        coalesce_columns(raw, "fase_ideal", ["fase_ideal", "fase_ideal_"], numeric=False)
        coalesce_columns(raw, "defasagem", ["defasagem", "defas"], numeric=True)
        coalesce_columns(raw, "nota_matematica", ["mat", "matem"], numeric=True)
        coalesce_columns(raw, "nota_portugues", ["por", "portug"], numeric=True)
        coalesce_columns(raw, "nota_ingles", ["ing", "ingles"], numeric=True)
        coalesce_columns(raw, "inde_ano", ["inde_2024", "inde_2023", "inde_23", "inde_22"], numeric=True)
        coalesce_columns(raw, "pedra_ano", ["pedra_2024", "pedra_2023", "pedra_23", "pedra_22"], numeric=False)
        coalesce_columns(raw, "ativo_inativo", ["ativo_inativo", "ativo_inativo_1"], numeric=False)

        for col in ["fase", "turma", "genero", "ian", "ida", "ieg", "iaa", "ips", "ipp", "ipv", "inde_22", "inde_23"]:
            if col not in raw.columns:
                raw[col] = np.nan

        phase_from_fase = raw["fase"].apply(extract_phase_code)
        phase_from_ideal = raw["fase_ideal"].apply(extract_phase_code)
        raw["fase_programa"] = phase_from_fase.where(phase_from_fase.notna(), phase_from_ideal)

        raw["idade"] = normalize_age_series(raw["idade"])

        for num_col in [
            "defasagem",
            "inde_22",
            "inde_23",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
        ]:
            raw[num_col] = coerce_numeric_series(raw[num_col])

        for score_col in [
            "inde_22",
            "inde_23",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
        ]:
            raw[score_col] = raw[score_col].clip(lower=0, upper=10)

        raw["genero"] = raw["genero"].apply(standardize_gender)
        raw["pedra_ano"] = raw["pedra_ano"].apply(standardize_stone)

        keep_cols = [
            "ra",
            "ano_referencia",
            "idade",
            "genero",
            "instituicao_ensino",
            "fase_programa",
            "turma",
            "pedra_ano",
            "inde_22",
            "inde_23",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "defasagem",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
            "ativo_inativo",
        ]
        for col in keep_cols:
            if col not in raw.columns:
                raw[col] = np.nan

        all_frames.append(raw[keep_cols].copy())

    if not all_frames:
        raise ValueError("Nenhuma aba valida foi encontrada para consolidacao.")

    df = pd.concat(all_frames, ignore_index=True)
    df = df.drop_duplicates(subset=["ra", "ano_referencia"], keep="first")
    df = df.sort_values(["ra", "ano_referencia"]).reset_index(drop=True)

    for cat_col in ["genero", "instituicao_ensino", "fase_programa", "turma", "pedra_ano", "ativo_inativo"]:
        df[cat_col] = clean_empty_strings(df[cat_col])

    return df


def load_consolidated_or_raw() -> pd.DataFrame:
    if PROCESSED_PATH.exists():
        df = pd.read_csv(PROCESSED_PATH)
        if "idade" in df.columns:
            df["idade"] = normalize_age_series(df["idade"])

        for num_col in [
            "defasagem",
            "inde_22",
            "inde_23",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
        ]:
            if num_col in df.columns:
                df[num_col] = coerce_numeric_series(df[num_col])
        return df

    if DATA_PATH.exists():
        return load_and_prepare_data(DATA_PATH)

    raise FileNotFoundError("Base consolidada e arquivo bruto indisponiveis para modelagem.")



## 3) Engenharia de target: risco de defasagem no ano subsequente

Aqui transformamos dados historicos em um problema supervisionado de classificacao binaria.

### O que e feito nesta celula
- Garante que as colunas esperadas existam.
- Cria duas visoes da base:
  - `base`: dados do ano atual (`ano_base`).
  - `future`: dados do ano seguinte por aluno (`ian_prox`, `ida_prox`, `inde_prox`).
- Faz `merge` por `ra` e ano para alinhar t (atual) com t+1 (futuro).

### Features derivadas criadas
- `delta_ian_prox`: variacao de IAN do ano atual para o proximo.
- `delta_ida_prox`: variacao de IDA.
- `delta_inde_hist`: variacao do INDE em relacao ao ano anterior do mesmo aluno.
- `media_notas`: media de matematica, portugues e ingles.
- `media_comportamental`: media de indicadores comportamentais.
- `desalinhamento_autoavaliacao`: diferenca entre autoavaliacao e indicadores de desempenho.

### Regra do target (variavel alvo)
`risco_defasagem_t1 = 1` se ocorrer pelo menos uma condicao:
- `IAN_t+1 <= 5.0`
- `IDA_t+1 <= 6.0`
- `delta_IAN_t+1 <= -1.0`

Caso contrario, `risco_defasagem_t1 = 0`.

### Recorte para treino/teste
A base rotulada usa anos-base 2022 e 2023 (pois nesses anos existe alvo conhecido no ano seguinte). Tambem mostramos distribuicao de classes para entender desbalanceamento.


In [3]:
df = load_consolidated_or_raw().copy()

expected_cols = [
    "ra",
    "ano_referencia",
    "idade",
    "genero",
    "instituicao_ensino",
    "fase_programa",
    "turma",
    "pedra_ano",
    "inde_22",
    "inde_23",
    "inde_ano",
    "ian",
    "ida",
    "ieg",
    "iaa",
    "ips",
    "ipp",
    "ipv",
    "defasagem",
    "nota_matematica",
    "nota_portugues",
    "nota_ingles",
    "ativo_inativo",
]
for col in expected_cols:
    if col not in df.columns:
        df[col] = np.nan

base = df.rename(columns={"ano_referencia": "ano_base"}).copy()
future = df[["ra", "ano_referencia", "ian", "ida", "inde_ano"]].copy()
future = future.rename(
    columns={
        "ano_referencia": "ano_base",
        "ian": "ian_prox",
        "ida": "ida_prox",
        "inde_ano": "inde_prox",
    }
)
future["ano_base"] = future["ano_base"] - 1

model_df = base.merge(future, on=["ra", "ano_base"], how="left")
model_df = model_df.sort_values(["ra", "ano_base"]).reset_index(drop=True)

model_df["delta_ian_prox"] = model_df["ian_prox"] - model_df["ian"]
model_df["delta_ida_prox"] = model_df["ida_prox"] - model_df["ida"]
model_df["delta_inde_hist"] = model_df["inde_ano"] - model_df.groupby("ra")["inde_ano"].shift(1)
model_df["media_notas"] = model_df[["nota_matematica", "nota_portugues", "nota_ingles"]].mean(axis=1)
model_df["media_comportamental"] = model_df[["iaa", "ieg", "ips", "ipp"]].mean(axis=1)
model_df["desalinhamento_autoavaliacao"] = model_df["iaa"] - model_df[["ida", "ieg"]].mean(axis=1)

model_df["target_disponivel"] = model_df["ian_prox"].notna() & model_df["ida_prox"].notna()
model_df["risco_defasagem_t1"] = (
    (model_df["ian_prox"] <= TARGET_IAN_THRESHOLD)
    | (model_df["delta_ian_prox"] <= -1.0)
).astype("Int64")

labeled_df = model_df[(model_df["target_disponivel"]) & (model_df["ano_base"].isin([2022, 2023]))].copy()

print("Base total:", df.shape)
print("Base rotulada para modelagem:", labeled_df.shape)
display(labeled_df[["ano_base", "risco_defasagem_t1"]].value_counts().rename("contagem").reset_index())

class_balance = (
    labeled_df.groupby("ano_base", as_index=False)["risco_defasagem_t1"]
    .mean()
    .rename(columns={"risco_defasagem_t1": "taxa_risco"})
)
class_balance["taxa_risco"] = class_balance["taxa_risco"] * 100
display(class_balance.round(2))

fig_balance = px.bar(
    class_balance,
    x="ano_base",
    y="taxa_risco",
    text_auto=".1f",
    title="Taxa de risco no target por ano-base",
    labels={"ano_base": "Ano-base", "taxa_risco": "Taxa de risco (%)"},
)
fig_balance.show()



Base total: (3027, 37)
Base rotulada para modelagem: (1267, 46)


,ano_base,risco_defasagem_t1,contagem
0,2023,0,385
1,2022,1,365
2,2023,1,308
3,2022,0,209


,ano_base,taxa_risco
0,2022,63.59
1,2023,44.44


## 4) Split temporal, preprocessamento e tratamento de desbalanceamento

Nesta etapa definimos como o modelo vai receber os dados.

### Split temporal (muito importante)
- **Treino**: ano-base 2022 (prediz 2023).
- **Teste temporal**: ano-base 2023 (prediz 2024).

Isso simula o uso real no tempo e evita vazamento de informacao do futuro para o passado.

### Selecao e poda de features (mitigacao de overfitting)
- Comecamos com listas de features numericas e categoricas.
- Depois removemos automaticamente colunas muito esparsas no treino.
- Isso reduz ruido, evita sinais instaveis e melhora generalizacao.

### Pipeline de preprocessamento
- Numericas:
  - `SimpleImputer(median)`: imputacao robusta para dados faltantes.
  - `StandardScaler`: coloca variaveis na mesma escala.
- Categoricas:
  - `SimpleImputer(most_frequent)`: preenche faltantes com categoria mais comum.
  - `OneHotEncoder`: transforma texto em colunas binarias.

Tudo isso e encapsulado em `ColumnTransformer`, garantindo preprocessamento consistente em treino e teste.

### Desbalanceamento
E calculado `scale_pos_weight` para o XGBoost, ajudando o modelo a tratar melhor a classe minoritaria (alunos em risco).


In [4]:
numeric_features_all = [
    "idade",
    "inde_22",
    "inde_23",
    "inde_ano",
    "ian",
    "ida",
    "ieg",
    "iaa",
    "ips",
    "ipp",
    "ipv",
    "defasagem",
    "nota_matematica",
    "nota_portugues",
    "nota_ingles",
    "media_notas",
    "media_comportamental",
    "desalinhamento_autoavaliacao",
    "delta_inde_hist",
]
categorical_features_all = ["genero", "instituicao_ensino", "fase_programa", "turma", "pedra_ano", "ativo_inativo"]

for col in numeric_features_all + categorical_features_all:
    if col not in labeled_df.columns:
        labeled_df[col] = np.nan

train_df = labeled_df[labeled_df["ano_base"] == 2022].copy()
test_df = labeled_df[labeled_df["ano_base"] == 2023].copy()
if train_df.empty or test_df.empty:
    raise ValueError("Split temporal invalido. Verifique os dados por ano-base.")

# Mitigacao de overfitting: removemos variaveis muito esparsas no treino.
NUMERIC_MIN_OBS_RATIO = 0.05
CATEGORICAL_MIN_OBS_RATIO = 0.05
MIN_CATEGORICAL_UNIQUE = 2

numeric_features = []
dropped_numeric = []
for col in numeric_features_all:
    observed_ratio = train_df[col].notna().mean()
    if observed_ratio >= NUMERIC_MIN_OBS_RATIO:
        numeric_features.append(col)
    else:
        dropped_numeric.append(col)

categorical_features = []
dropped_categorical = []
for col in categorical_features_all:
    observed_ratio = train_df[col].notna().mean()
    unique_values = train_df[col].dropna().astype(str).nunique()
    if observed_ratio >= CATEGORICAL_MIN_OBS_RATIO and unique_values >= MIN_CATEGORICAL_UNIQUE:
        categorical_features.append(col)
    else:
        dropped_categorical.append(col)

feature_columns = numeric_features + categorical_features
if not feature_columns:
    raise ValueError("Nenhuma feature valida apos filtragem de esparsidade.")

X_train = train_df[feature_columns].copy()
y_train = train_df["risco_defasagem_t1"].astype(int)
X_test = test_df[feature_columns].copy()
y_test = test_df["risco_defasagem_t1"].astype(int)

scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))

print("Treino:", X_train.shape, "| taxa positiva:", round(y_train.mean(), 3))
print("Teste:", X_test.shape, "| taxa positiva:", round(y_test.mean(), 3))
print("Features numericas mantidas:", len(numeric_features), "| removidas:", dropped_numeric)
print("Features categoricas mantidas:", len(categorical_features), "| removidas:", dropped_categorical)
print("scale_pos_weight para XGBoost:", round(scale_pos_weight, 3))

# Imputacao por mediana tende a generalizar melhor em amostras pequenas do que KNN.
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)


Treino: (574, 21) | taxa positiva: 0.636
Teste: (693, 21) | taxa positiva: 0.444
Features numericas mantidas: 16 | removidas: ['inde_23', 'ipp', 'delta_inde_hist']
Features categoricas mantidas: 5 | removidas: ['ativo_inativo']
scale_pos_weight para XGBoost: 0.573


## 5) Treino e comparacao de modelos

Esta secao executa o experimento principal.

### Modelos avaliados
- Regressao Logistica (baseline simples e interpretavel).
- Random Forest (regularizado).
- Extra Trees (regularizado).
- XGBoost (regularizado).

### Mitigacao aplicada nesta fase
- Reducao de complexidade das arvores (profundidade menor e folhas maiores).
- Maior regularizacao no XGBoost (`reg_alpha`, `reg_lambda`, `gamma`, `min_child_weight`).
- Mantida otimizacao de threshold com foco em Recall.

### Estrategia de validacao
Para cada modelo:
1. Monta um `Pipeline` = preprocessador + algoritmo.
2. Roda validacao cruzada estratificada (`StratifiedKFold`) no treino.
3. Calcula metricas de CV (accuracy, ROC-AUC, PR-AUC, recall, F1).
4. Gera probabilidades OOF (`cross_val_predict`) para escolher threshold.

### Ajuste de threshold orientado a negocio
- Testa varios cortes de probabilidade (`THRESHOLD_GRID`).
- Filtra cortes com `recall >= RECALL_OBJECTIVE` (0.80).
- Entre os validos, escolhe o melhor por F1 (desempate por precision, recall e accuracy).

Depois disso, o modelo e treinado em todo treino e avaliado no teste temporal.

### Saidas desta secao
- `cv_metrics_table`: desempenho em validacao cruzada.
- `metrics_table`: desempenho no teste temporal para comparar modelos de forma justa.


In [5]:
models = {
    # Configuracoes regularizadas para reduzir overfitting.
    "Regressao Logistica": LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        C=0.5,
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=6,
        min_samples_split=30,
        min_samples_leaf=14,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=450,
        max_depth=6,
        min_samples_split=32,
        min_samples_leaf=16,
        max_features="sqrt",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=350,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        min_child_weight=10,
        gamma=1.5,
        reg_alpha=1.2,
        reg_lambda=6.0,
        max_delta_step=1,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=1,
        tree_method="hist",
    ),
}


def evaluate_threshold_grid(y_true: pd.Series, y_prob: np.ndarray, thresholds: np.ndarray) -> pd.DataFrame:
    rows = []
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        rows.append(
            {
                "threshold": float(threshold),
                "accuracy": accuracy_score(y_true, y_pred),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "f1": f1_score(y_true, y_pred, zero_division=0),
            }
        )
    return pd.DataFrame(rows)


def choose_threshold(threshold_df: pd.DataFrame, recall_floor: float) -> float:
    valid = threshold_df[threshold_df["recall"] >= recall_floor].copy()
    if valid.empty:
        valid = threshold_df.copy()
    winner = valid.sort_values(["f1", "precision", "recall", "accuracy"], ascending=False).iloc[0]
    return float(winner["threshold"])


def evaluate_probabilities(y_true: pd.Series, y_prob: np.ndarray, threshold: float) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "y_pred": y_pred,
        "report": classification_report(y_true, y_pred, output_dict=True, zero_division=0),
    }


cv_scoring = {
    "accuracy": "accuracy",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "recall": "recall",
    "f1": "f1",
}

results = {}
cv_rows = []

for model_name, estimator in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )

    cv_scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=cv_scoring,
        n_jobs=1,
        return_train_score=False,
    )

    cv_rows.append(
        {
            "modelo": model_name,
            "cv_accuracy_mean": cv_scores["test_accuracy"].mean(),
            "cv_accuracy_std": cv_scores["test_accuracy"].std(),
            "cv_roc_auc_mean": cv_scores["test_roc_auc"].mean(),
            "cv_roc_auc_std": cv_scores["test_roc_auc"].std(),
            "cv_pr_auc_mean": cv_scores["test_pr_auc"].mean(),
            "cv_pr_auc_std": cv_scores["test_pr_auc"].std(),
            "cv_recall_mean": cv_scores["test_recall"].mean(),
            "cv_f1_mean": cv_scores["test_f1"].mean(),
        }
    )

    oof_prob = cross_val_predict(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=1,
    )[:, 1]
    threshold_table = evaluate_threshold_grid(y_train, oof_prob, THRESHOLD_GRID)
    threshold = choose_threshold(threshold_table, recall_floor=RECALL_OBJECTIVE)

    pipeline.fit(X_train, y_train)
    test_prob = pipeline.predict_proba(X_test)[:, 1]
    metrics = evaluate_probabilities(y_test, test_prob, threshold)

    results[model_name] = {
        "pipeline": pipeline,
        "threshold": threshold,
        "metrics": metrics,
        "test_prob": test_prob,
        "oof_prob": oof_prob,
        "threshold_table": threshold_table,
    }

cv_metrics_table = pd.DataFrame(cv_rows).sort_values(["cv_pr_auc_mean", "cv_recall_mean"], ascending=False)
metrics_table = pd.DataFrame(
    [
        {
            "modelo": name,
            "threshold": result["threshold"],
            "accuracy": result["metrics"]["accuracy"],
            "precision": result["metrics"]["precision"],
            "recall": result["metrics"]["recall"],
            "f1": result["metrics"]["f1"],
            "roc_auc": result["metrics"]["roc_auc"],
            "pr_auc": result["metrics"]["pr_auc"],
            "brier": result["metrics"]["brier"],
        }
        for name, result in results.items()
    ]
).sort_values(["pr_auc", "recall", "roc_auc"], ascending=False)

print("Resumo de validacao cruzada (treino 2022):")
display(cv_metrics_table.round(4))

print("Resumo no teste temporal (2023):")
display(metrics_table.round(4))


Resumo de validacao cruzada (treino 2022):


,modelo,cv_accuracy_mean,cv_accuracy_std,cv_roc_auc_mean,cv_roc_auc_std,cv_pr_auc_mean,cv_pr_auc_std,cv_recall_mean,cv_f1_mean
0,Regressao Logistica,0.7753,0.0451,0.8575,0.0553,0.9143,0.0369,0.7781,0.8148
1,Random Forest,0.7543,0.0493,0.8362,0.0503,0.9017,0.0304,0.7808,0.8013
2,Extra Trees,0.7543,0.0440,0.8355,0.0547,0.8999,0.0355,0.7589,0.7967
3,XGBoost,0.7526,0.0556,0.8263,0.0440,0.8939,0.0302,0.7644,0.7970


Resumo no teste temporal (2023):


,modelo,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,brier
3,XGBoost,0.250,0.6869,0.5987,0.8961,0.7178,0.8450,0.8262,0.1620
0,Regressao Logistica,0.325,0.7374,0.7351,0.6396,0.6840,0.8317,0.8134,0.1835
1,Random Forest,0.325,0.6753,0.5889,0.8929,0.7097,0.8278,0.8129,0.1750
2,Extra Trees,0.325,0.6580,0.5767,0.8669,0.6926,0.8104,0.7854,0.1905


## 6) Avaliacao: matriz de confusao, ROC-AUC e analise de acuracia

Aqui fazemos diagnostico detalhado da qualidade do modelo.

### O que e analisado
- **Matriz de confusao** por modelo:
  - verdadeiros positivos, falsos positivos, verdadeiros negativos, falsos negativos.
- **Curva ROC** e AUC:
  - capacidade de separar classes em varios thresholds.
- **Curva Precision-Recall**:
  - especialmente util quando ha desbalanceamento.
- **Curva de threshold** do melhor modelo:
  - mostra trade-off entre precision, recall e F1.
- **Tabela de calibracao por faixas**:
  - compara probabilidade prevista vs taxa real observada.

### Analise de acuracia para todos os modelos
Tambem e criada uma comparacao entre:
- acuracia no teste temporal,
- acuracia media na validacao cruzada,
- diferenca entre teste e CV (estabilidade).

Isso ajuda a identificar overfitting e consistencia de generalizacao.


In [6]:
for model_name, result in results.items():
    y_pred = result["metrics"]["y_pred"]
    cm = confusion_matrix(y_test, y_pred)
    fig_cm = go.Figure(
        data=go.Heatmap(
            z=cm,
            x=["Pred 0", "Pred 1"],
            y=["Real 0", "Real 1"],
            text=cm,
            texttemplate="%{text}",
            colorscale="Blues",
        )
    )
    fig_cm.update_layout(title=f"Matriz de confusao - {model_name}")
    fig_cm.show()

roc_fig = go.Figure()
pr_fig = go.Figure()
for model_name, result in results.items():
    fpr, tpr, _ = roc_curve(y_test, result["test_prob"])
    precision, recall, _ = precision_recall_curve(y_test, result["test_prob"])

    roc_fig.add_trace(
        go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{model_name} (AUC={result['metrics']['roc_auc']:.3f})",
        )
    )

    pr_fig.add_trace(
        go.Scatter(
            x=recall,
            y=precision,
            mode="lines",
            name=f"{model_name} (PR-AUC={result['metrics']['pr_auc']:.3f})",
        )
    )

roc_fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Aleatorio", line=dict(dash="dash")))
roc_fig.update_layout(
    title="Curva ROC - comparacao de modelos",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    yaxis=dict(scaleanchor="x", scaleratio=1),
    xaxis=dict(constrain="domain"),
)
roc_fig.show()

pr_fig.update_layout(
    title="Curva Precision-Recall - comparacao de modelos",
    xaxis_title="Recall",
    yaxis_title="Precision",
)
pr_fig.show()

best_model_name_eval = metrics_table.iloc[0]["modelo"]
best_result_eval = results[best_model_name_eval]

threshold_curve_best = best_result_eval["threshold_table"].copy()
fig_threshold = px.line(
    threshold_curve_best,
    x="threshold",
    y=["precision", "recall", "f1"],
    markers=True,
    title=f"Curva de threshold (OOF treino) - {best_model_name_eval}",
    labels={"value": "Score", "threshold": "Threshold"},
)
fig_threshold.add_vline(
    x=best_result_eval["threshold"],
    line_dash="dash",
    line_color="red",
    annotation_text=f"Threshold selecionado: {best_result_eval['threshold']:.2f}",
)
fig_threshold.show()

calibration_df = pd.DataFrame({"prob": best_result_eval["test_prob"], "real": y_test.values})
calibration_df["faixa_prob"] = pd.cut(calibration_df["prob"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
calibration_table = (
    calibration_df.groupby("faixa_prob", as_index=False)
    .agg(amostra=("real", "size"), taxa_real=("real", "mean"), prob_media=("prob", "mean"))
)
calibration_table["taxa_real"] = calibration_table["taxa_real"] * 100
calibration_table["prob_media"] = calibration_table["prob_media"] * 100

print(f"Calibracao por faixas - {best_model_name_eval}")
display(calibration_table.round(2))

# Analise comparativa de acuracia para todos os modelos
accuracy_analysis = (
    metrics_table[["modelo", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]]
    .merge(
        cv_metrics_table[["modelo", "cv_accuracy_mean", "cv_accuracy_std", "cv_recall_mean", "cv_f1_mean"]],
        on="modelo",
        how="left",
    )
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)
accuracy_analysis["rank_acuracia_teste"] = accuracy_analysis["accuracy"].rank(method="min", ascending=False).astype(int)
accuracy_analysis["delta_accuracy_teste_vs_cv"] = accuracy_analysis["accuracy"] - accuracy_analysis["cv_accuracy_mean"]

print("Analise de acuracia e estabilidade (CV x teste):")
display(accuracy_analysis.round(4))

accuracy_plot_df = accuracy_analysis.melt(
    id_vars=["modelo"],
    value_vars=["accuracy", "cv_accuracy_mean"],
    var_name="origem",
    value_name="acuracia",
)
accuracy_plot_df["origem"] = accuracy_plot_df["origem"].map(
    {"accuracy": "Acuracia no teste temporal", "cv_accuracy_mean": "Acuracia media CV (treino)"}
)

fig_accuracy = px.bar(
    accuracy_plot_df,
    x="modelo",
    y="acuracia",
    color="origem",
    barmode="group",
    text=accuracy_plot_df["acuracia"].map(lambda v: f"{v:.3f}"),
    title="Comparacao de acuracia por modelo (teste temporal vs CV)",
    labels={"modelo": "Modelo", "acuracia": "Acuracia", "origem": "Origem da metrica"},
)
fig_accuracy.update_yaxes(range=[0, 1])
fig_accuracy.update_traces(textposition="outside")
fig_accuracy.show()

best_acc = accuracy_analysis.iloc[0]
worst_acc = accuracy_analysis.iloc[-1]

analysis_lines = [
    "### Analise de acuracia de todos os modelos",
    "",
    f"- Melhor acuracia no teste temporal: **{best_acc['modelo']}** ({best_acc['accuracy']:.3f}).",
    f"- Menor acuracia no teste temporal: **{worst_acc['modelo']}** ({worst_acc['accuracy']:.3f}).",
    f"- Diferenca entre melhor e pior acuracia: **{(best_acc['accuracy'] - worst_acc['accuracy']):.3f}**.",
    "",
    "Detalhamento por modelo:",
]

for _, row in accuracy_analysis.iterrows():
    analysis_lines.append(
        f"- **{row['modelo']}**: teste={row['accuracy']:.3f}, CV={row['cv_accuracy_mean']:.3f}, "
        f"delta_teste_cv={row['delta_accuracy_teste_vs_cv']:+.3f}, recall_teste={row['recall']:.3f}."
    )

display(Markdown("\n".join(analysis_lines)))





Calibracao por faixas - XGBoost


,faixa_prob,amostra,taxa_real,prob_media
0,"(-0.001, 0.2]",194,7.73,11.460000
1,"(0.2, 0.4]",192,39.06,31.000000
2,"(0.4, 0.6]",128,49.22,49.820000
3,"(0.6, 0.8]",114,80.70,69.779999
4,"(0.8, 1.0]",65,96.92,86.370003


Analise de acuracia e estabilidade (CV x teste):


,modelo,accuracy,precision,recall,f1,roc_auc,pr_auc,cv_accuracy_mean,cv_accuracy_std,cv_recall_mean,cv_f1_mean,rank_acuracia_teste,delta_accuracy_teste_vs_cv
0,Regressao Logistica,0.7374,0.7351,0.6396,0.6840,0.8317,0.8134,0.7753,0.0451,0.7781,0.8148,1,-0.0379
1,XGBoost,0.6869,0.5987,0.8961,0.7178,0.8450,0.8262,0.7526,0.0556,0.7644,0.7970,2,-0.0657
2,Random Forest,0.6753,0.5889,0.8929,0.7097,0.8278,0.8129,0.7543,0.0493,0.7808,0.8013,3,-0.0790
3,Extra Trees,0.6580,0.5767,0.8669,0.6926,0.8104,0.7854,0.7543,0.0440,0.7589,0.7967,4,-0.0963


### Analise de acuracia de todos os modelos

- Melhor acuracia no teste temporal: **Regressao Logistica** (0.737).
- Menor acuracia no teste temporal: **Extra Trees** (0.658).
- Diferenca entre melhor e pior acuracia: **0.079**.

Detalhamento por modelo:
- **Regressao Logistica**: teste=0.737, CV=0.775, delta_teste_cv=-0.038, recall_teste=0.640.
- **XGBoost**: teste=0.687, CV=0.753, delta_teste_cv=-0.066, recall_teste=0.896.
- **Random Forest**: teste=0.675, CV=0.754, delta_teste_cv=-0.079, recall_teste=0.893.
- **Extra Trees**: teste=0.658, CV=0.754, delta_teste_cv=-0.096, recall_teste=0.867.

## 7) Explicabilidade: Feature Importance por modelo (e SHAP opcional)

Depois de medir desempenho, explicamos **por que** o modelo preve risco.

### Etapas desta secao
- Recupera nomes das features apos preprocessamento (incluindo one-hot).
- Mapeia features transformadas para suas features originais.
- Calcula importancia por modelo:
  - arvore: `feature_importances_`.
  - regressao: modulo dos coeficientes.
- Normaliza importancias para facilitar comparacao.

### Analises geradas
- Top 10 features por modelo.
- Heatmap com features de consenso entre modelos.
- Relatorio textual com as variaveis mais relevantes na media.
- Detalhamento do melhor modelo no teste temporal.

### SHAP (opcional)
Se a biblioteca `shap` estiver disponivel, calcula contribuicoes locais/globais mais detalhadas. Se nao estiver, o notebook segue sem quebrar.

Para iniciantes: pense no SHAP como uma forma de dizer "quanto cada variavel empurrou a previsao para cima ou para baixo".


In [7]:
def map_transformed_to_original_feature(feature_name: str, categorical_cols: list[str]) -> str:
    if feature_name.startswith("num__"):
        return feature_name.replace("num__", "", 1)

    if feature_name.startswith("cat__"):
        raw = feature_name.replace("cat__", "", 1)
        for cat_col in sorted(categorical_cols, key=len, reverse=True):
            prefix = f"{cat_col}_"
            if raw.startswith(prefix):
                return cat_col
        return raw

    return feature_name


# Importancia de features para todos os modelos
all_importance_frames = []
for model_name, model_result in results.items():
    pipeline_model = model_result["pipeline"]
    fitted_preprocessor_model = pipeline_model.named_steps["preprocessor"]
    fitted_estimator_model = pipeline_model.named_steps["model"]
    feature_names_model = fitted_preprocessor_model.get_feature_names_out()

    if hasattr(fitted_estimator_model, "feature_importances_"):
        raw_importance_values = np.asarray(fitted_estimator_model.feature_importances_, dtype=float)
    elif hasattr(fitted_estimator_model, "coef_"):
        raw_importance_values = np.abs(np.asarray(fitted_estimator_model.coef_).reshape(-1)).astype(float)
    else:
        continue

    model_importance_df = pd.DataFrame(
        {
            "feature_transformada": feature_names_model,
            "importancia_bruta": raw_importance_values,
        }
    )
    model_importance_df["feature_original"] = model_importance_df["feature_transformada"].apply(
        lambda x: map_transformed_to_original_feature(x, categorical_features)
    )

    model_importance_grouped = (
        model_importance_df.groupby("feature_original", as_index=False)["importancia_bruta"]
        .sum()
        .sort_values("importancia_bruta", ascending=False)
    )

    total_importance = model_importance_grouped["importancia_bruta"].sum()
    model_importance_grouped["importancia_normalizada"] = (
        model_importance_grouped["importancia_bruta"] / total_importance if total_importance > 0 else 0.0
    )
    model_importance_grouped["modelo"] = model_name
    all_importance_frames.append(model_importance_grouped)

all_models_importance_df = pd.concat(all_importance_frames, ignore_index=True)

model_importance_top10 = (
    all_models_importance_df.sort_values(["modelo", "importancia_normalizada"], ascending=[True, False])
    .groupby("modelo", as_index=False)
    .head(10)
)

print("Top 10 features por modelo (importancia normalizada):")
display(
    model_importance_top10[["modelo", "feature_original", "importancia_normalizada"]]
    .sort_values(["modelo", "importancia_normalizada"], ascending=[True, False])
    .round(4)
)

fig_importance_models = px.bar(
    model_importance_top10.sort_values(["modelo", "importancia_normalizada"], ascending=[True, True]),
    x="importancia_normalizada",
    y="feature_original",
    color="modelo",
    facet_col="modelo",
    facet_col_wrap=2,
    orientation="h",
    title="Top 10 features por modelo (importancia normalizada)",
    labels={
        "importancia_normalizada": "Importancia normalizada",
        "feature_original": "Feature original",
        "modelo": "Modelo",
    },
)
fig_importance_models.update_layout(showlegend=False, height=900)
fig_importance_models.show()

consensus_feature_list = (
    all_models_importance_df.groupby("feature_original")["importancia_normalizada"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
    .index.tolist()
)

importance_heatmap_df = (
    all_models_importance_df[all_models_importance_df["feature_original"].isin(consensus_feature_list)]
    .pivot_table(index="feature_original", columns="modelo", values="importancia_normalizada", aggfunc="mean")
    .fillna(0)
)
importance_heatmap_df = importance_heatmap_df.loc[
    importance_heatmap_df.mean(axis=1).sort_values(ascending=True).index
]

print("Importancia normalizada (features consenso) por modelo:")
display(importance_heatmap_df.round(4))

fig_importance_heatmap = px.imshow(
    importance_heatmap_df,
    text_auto=".3f",
    aspect="auto",
    color_continuous_scale="Blues",
    title="Mapa de importancia das features nos modelos preditivos",
    labels={"x": "Modelo", "y": "Feature original", "color": "Importancia normalizada"},
)
fig_importance_heatmap.show()

# Analise textual das features mais relevantes
feature_consensus = (
    all_models_importance_df.groupby("feature_original")["importancia_normalizada"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

consensus_lines = [
    "### Analise de features importantes nos modelos preditivos",
    "",
    "Features com maior importancia media entre os modelos:",
]
for _, row in feature_consensus.head(8).iterrows():
    consensus_lines.append(f"- **{row['feature_original']}**: importancia media = {row['importancia_normalizada']:.3f}")

display(Markdown("\n".join(consensus_lines)))

# Explicabilidade detalhada do melhor modelo
best_model_name = metrics_table.iloc[0]["modelo"]
best_pipeline = results[best_model_name]["pipeline"]
best_threshold = results[best_model_name]["threshold"]

print("Melhor modelo (teste temporal):", best_model_name)
print("Threshold otimizado no treino (OOF):", round(best_threshold, 4))

best_importance_df = (
    all_models_importance_df[all_models_importance_df["modelo"] == best_model_name]
    .sort_values("importancia_normalizada", ascending=False)
    .reset_index(drop=True)
)

importance_df = best_importance_df.rename(
    columns={"feature_original": "feature", "importancia_normalizada": "importance"}
)[["feature", "importance", "importancia_bruta", "modelo"]]

display(importance_df.head(25).round(4))

fig_importance = px.bar(
    importance_df.head(20).sort_values("importance", ascending=True),
    x="importance",
    y="feature",
    orientation="h",
    title=f"Top 20 fatores mais relevantes - {best_model_name}",
    labels={"importance": "Importancia normalizada", "feature": "Feature"},
)
fig_importance.show()

report_df = pd.DataFrame(results[best_model_name]["metrics"]["report"]).T
print("Relatorio de classificacao (teste 2023):")
display(report_df.round(4))

try:
    import shap

    fitted_preprocessor = best_pipeline.named_steps["preprocessor"]
    fitted_model = best_pipeline.named_steps["model"]
    feature_names = fitted_preprocessor.get_feature_names_out()

    X_test_sample = X_test.sample(min(250, len(X_test)), random_state=RANDOM_STATE)
    X_test_transformed = fitted_preprocessor.transform(X_test_sample)
    if hasattr(X_test_transformed, "toarray"):
        X_test_transformed = X_test_transformed.toarray()

    if hasattr(fitted_model, "feature_importances_"):
        explainer = shap.TreeExplainer(fitted_model)
        shap_values = explainer.shap_values(X_test_transformed)
        if isinstance(shap_values, list):
            shap_matrix = shap_values[1]
        else:
            shap_matrix = shap_values
    else:
        explainer = shap.LinearExplainer(fitted_model, X_test_transformed)
        shap_matrix = explainer.shap_values(X_test_transformed)

    mean_abs_shap = np.abs(shap_matrix).mean(axis=0)
    shap_df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs_shap}).sort_values("mean_abs_shap", ascending=False)
    display(shap_df.head(15))

    fig_shap = px.bar(
        shap_df.head(15).sort_values("mean_abs_shap", ascending=True),
        x="mean_abs_shap",
        y="feature",
        orientation="h",
        title="Top 15 features por |SHAP| medio",
        labels={"mean_abs_shap": "|SHAP| medio", "feature": "Feature"},
    )
    fig_shap.show()
except Exception as shap_error:
    print("SHAP nao executado (mantendo apenas feature importance):", shap_error)




Top 10 features por modelo (importancia normalizada):


,modelo,feature_original,importancia_normalizada
42,Extra Trees,ian,0.1702
43,Extra Trees,pedra_ano,0.1547
44,Extra Trees,instituicao_ensino,0.1319
45,Extra Trees,defasagem,0.1201
46,Extra Trees,fase_programa,0.1174
47,Extra Trees,inde_22,0.0633
48,Extra Trees,inde_ano,0.0524
49,Extra Trees,genero,0.0464
50,Extra Trees,ipv,0.0344
51,Extra Trees,idade,0.0189


Importancia normalizada (features consenso) por modelo:


modelo,Extra Trees,Random Forest,Regressao Logistica,XGBoost
feature_original,,,,
ieg,0.0083,0.0241,0.0151,0.0332
nota_matematica,0.0136,0.0316,0.0141,0.0252
nota_portugues,0.0087,0.0288,0.0129,0.0389
media_notas,0.0141,0.0405,0.0136,0.0369
idade,0.0189,0.0470,0.0370,0.0275
genero,0.0464,0.0256,0.0306,0.0592
ipv,0.0344,0.0658,0.0209,0.0621
inde_ano,0.0524,0.0907,0.0266,0.0590
inde_22,0.0633,0.0922,0.0266,0.0574


### Analise de features importantes nos modelos preditivos

Features com maior importancia media entre os modelos:
- **fase_programa**: importancia media = 0.131
- **defasagem**: importancia media = 0.117
- **turma**: importancia media = 0.101
- **ian**: importancia media = 0.100
- **instituicao_ensino**: importancia media = 0.085
- **pedra_ano**: importancia media = 0.068
- **inde_22**: importancia media = 0.060
- **inde_ano**: importancia media = 0.057

Melhor modelo (teste temporal): XGBoost
Threshold otimizado no treino (OOF): 0.25


,feature,importance,importancia_bruta,modelo
0,instituicao_ensino,0.1304,0.1304,XGBoost
1,defasagem,0.1172,0.1172,XGBoost
2,fase_programa,0.1113,0.1113,XGBoost
3,ian,0.0938,0.0938,XGBoost
4,ipv,0.0621,0.0621,XGBoost
5,genero,0.0592,0.0592,XGBoost
6,inde_ano,0.0590,0.0590,XGBoost
7,inde_22,0.0574,0.0574,XGBoost
8,nota_portugues,0.0389,0.0389,XGBoost
9,media_notas,0.0369,0.0369,XGBoost


Relatorio de classificacao (teste 2023):


,precision,recall,f1-score,support
0,0.8621,0.5195,0.6483,385.0000
1,0.5987,0.8961,0.7178,308.0000
accuracy,0.6869,0.6869,0.6869,0.6869
macro avg,0.7304,0.7078,0.6831,693.0000
weighted avg,0.7450,0.6869,0.6792,693.0000


,feature,mean_abs_shap
3,num__ian,0.558152
8,num__ipv,0.412648
9,num__defasagem,0.250975
21,cat__fase_programa_ALFA,0.225420
16,cat__genero_Feminino,0.169472
1,num__inde_22,0.163078
0,num__idade,0.127883
14,num__media_comportamental,0.114524
2,num__inde_ano,0.096342
10,num__nota_matematica,0.077159


## 8) Exportacao de artefatos (modelo, scaler e score 2024)

Etapa final: empacotar o que foi construido para uso em producao ou analise posterior.

### O que e salvo
- `modelo_risco_defasagem.pkl`: bundle completo com pipeline, threshold, metadados e comparacoes.
- `scaler_numerico.pkl`: scaler das variaveis numericas.
- `feature_importance.csv`: importancia do melhor modelo.
- `feature_importance_all_models.csv`: importancia de todos os modelos.
- `model_comparison_test.csv`: comparacao final no teste.
- `model_accuracy_analysis.csv`: analise de acuracia vs CV.
- `threshold_curve_best_model.csv`: curva de corte do melhor modelo.

### Scoring 2024
Se houver dados de `ano_base == 2024`, o notebook:
- calcula probabilidade de risco,
- aplica threshold para classe prevista,
- classifica nivel de risco (`Baixo`, `Moderado`, `Alto`),
- exporta `predicoes_risco_2024.csv`.

### Resultado pratico
Com esses artefatos, voce consegue reaproveitar o modelo sem retreinar toda vez e integrar a previsao em app, API ou dashboard.


In [8]:
default_numeric = {
    col: float(labeled_df[col].median()) if labeled_df[col].notna().any() else 0.0
    for col in numeric_features
}
default_categorical = {}
for col in categorical_features:
    mode_series = labeled_df[col].dropna().astype(str)
    default_categorical[col] = mode_series.mode().iloc[0] if not mode_series.empty else "Nao Informado"

category_options = {col: sorted(labeled_df[col].dropna().astype(str).unique().tolist()) for col in categorical_features}

best_metrics = results[best_model_name]["metrics"]
risk_bands = {
    "baixo_max": float(max(0.20, best_threshold - 0.15)),
    "alto_min": float(best_threshold),
}

training_info = {
    "train_years": [2022],
    "test_years": [2023],
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "target_positive_rate_train": float(y_train.mean()),
    "target_positive_rate_test": float(y_test.mean()),
    "best_model_test_metrics": {
        "accuracy": float(best_metrics["accuracy"]),
        "precision": float(best_metrics["precision"]),
        "recall": float(best_metrics["recall"]),
        "f1": float(best_metrics["f1"]),
        "roc_auc": float(best_metrics["roc_auc"]),
        "pr_auc": float(best_metrics["pr_auc"]),
        "brier": float(best_metrics["brier"]),
    },
}

bundle = {
    "model_name": best_model_name,
    "pipeline": best_pipeline,
    "threshold": float(best_threshold),
    "risk_bands": risk_bands,
    "feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target_rule": {
        "description": "risco=1 se (IAN_t+1 <= 5.0) OU (IDA_t+1 <= 6.0) OU (queda_ian_t+1 <= -1.0)",
        "ian_threshold": TARGET_IAN_THRESHOLD,
        "delta_ian_threshold": -1.0,
    },
    "healthy_thresholds": {"ian": 5.0, "ida": 6.0, "ieg": 7.0, "ips": 7.0, "ipp": 7.0, "ipv": 7.0},
    "default_inputs": {**default_numeric, **default_categorical},
    "category_options": category_options,
    "training_info": training_info,
    "model_comparison_cv": cv_metrics_table.to_dict(orient="records"),
    "model_comparison_test": metrics_table.to_dict(orient="records"),
    "threshold_curve_train_cv": results[best_model_name]["threshold_table"].to_dict(orient="records"),
    "feature_importance_best_model": importance_df.to_dict(orient="records"),
    "feature_importance_all_models": all_models_importance_df.to_dict(orient="records"),
}

model_path = ARTIFACT_DIR / "modelo_risco_defasagem.pkl"
with model_path.open("wb") as file:
    pickle.dump(bundle, file)

scaler_obj = best_pipeline.named_steps["preprocessor"].named_transformers_["num"].named_steps["scaler"]
scaler_path = ARTIFACT_DIR / "scaler_numerico.pkl"
with scaler_path.open("wb") as file:
    pickle.dump(scaler_obj, file)

importance_path = ARTIFACT_DIR / "feature_importance.csv"
importance_df.to_csv(importance_path, index=False)

importance_all_path = ARTIFACT_DIR / "feature_importance_all_models.csv"
all_models_importance_df.to_csv(importance_all_path, index=False)

model_compare_path = ARTIFACT_DIR / "model_comparison_test.csv"
metrics_table.to_csv(model_compare_path, index=False)

model_compare_accuracy_path = ARTIFACT_DIR / "model_accuracy_analysis.csv"
accuracy_analysis.to_csv(model_compare_accuracy_path, index=False)

threshold_curve_path = ARTIFACT_DIR / "threshold_curve_best_model.csv"
results[best_model_name]["threshold_table"].to_csv(threshold_curve_path, index=False)

score_2024 = model_df[model_df["ano_base"] == 2024].copy()
if not score_2024.empty:
    X_2024 = score_2024[feature_columns].copy()
    score_2024["prob_risco"] = best_pipeline.predict_proba(X_2024)[:, 1]
    score_2024["risco_predito"] = (score_2024["prob_risco"] >= best_threshold).astype(int)
    score_2024["nivel_risco"] = pd.cut(
        score_2024["prob_risco"],
        bins=[-0.01, risk_bands["baixo_max"], risk_bands["alto_min"], 1.0],
        labels=["Baixo", "Moderado", "Alto"],
    )
    score_2024 = score_2024.sort_values("prob_risco", ascending=False)
    score_2024_out = ARTIFACT_DIR / "predicoes_risco_2024.csv"
    score_2024[["ra", "ano_base", "prob_risco", "risco_predito", "nivel_risco"]].to_csv(score_2024_out, index=False)

print("Artefatos exportados:")
print("-", model_path.resolve())
print("-", scaler_path.resolve())
print("-", importance_path.resolve())
print("-", importance_all_path.resolve())
print("-", model_compare_path.resolve())
print("-", model_compare_accuracy_path.resolve())
print("-", threshold_curve_path.resolve())
if not score_2024.empty:
    print("-", score_2024_out.resolve())




Artefatos exportados:
- C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\artifacts\modelo_risco_defasagem.pkl
- C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\artifacts\scaler_numerico.pkl
- C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\artifacts\feature_importance.csv
- C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\artifacts\feature_importance_all_models.csv
- C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\artifacts\model_comparison_test.csv
- C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\artifacts\model_accuracy_analysis.csv
- C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\artifacts\threshold_curve_best_model.csv
- C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\artifacts\predicoes_risco_2024.csv
